### INGEST

In [0]:
dbutils.fs.ls('/Volumes/workspace/default/my_files')

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format('csv').option('header', 'true').load('/Volumes/workspace/default/my_files/KaggleV2-May-2016.csv')
df.display()

In [0]:
my_struct_schema = StructType([
  StructField('PatientId', LongType(), True),
  StructField('AppointmentID', LongType(), True),
  StructField('Gender', StringType(), True),
  StructField('ScheduledDay', TimestampType(), True),
  StructField('AppointmentDay', TimestampType(), True),
  StructField('Age', IntegerType(), True),
  StructField('Neighbourhood', StringType(), True),
  StructField('Scholarship', IntegerType(), True),
  StructField('Hipertension', IntegerType(), True),
  StructField('Diabetes', IntegerType(), True),
  StructField('Alcoholism', IntegerType(), True),
  StructField('Handcap', IntegerType(), True),
  StructField('SMS_received', IntegerType(), True),
  StructField('No-show', StringType(), True)
])

In [0]:
df = spark.read.format('csv').schema(my_struct_schema).option('header', 'true').load('/Volumes/workspace/default/my_files/KaggleV2-May-2016.csv')
df.display()

In [0]:
output_path = "/Volumes/workspace/default/my_files/medical_appointment/ingest"

df.write.mode("overwrite").parquet(output_path)

### Clean and transform

In [0]:
df = df.withColumnRenamed('PatientId', 'Patient_ID')\
    .withColumnRenamed('AppointmentID', 'Appointment_ID')\
    .withColumnRenamed('Gender', 'Gender')\
    .withColumnRenamed('ScheduledDay', 'Scheduled_Day')\
    .withColumnRenamed('AppointmentDay', 'Appointment_Day')\
    .withColumnRenamed('Age', 'Age')\
    .withColumnRenamed('Neighbourhood', 'Neighbourhood')\
    .withColumnRenamed('No-show', 'NoShow')\
    .withColumnRenamed('Hipertension', 'Hypertension')\
    .withColumnRenamed('SMS_received', 'SMS_Received')\
    .withColumnRenamed('Scholarship', 'Scholarship')\
    .withColumnRenamed('Diabetes', 'Diabetes')\
    .withColumnRenamed('Alcoholism', 'Alcoholism')\
    .withColumnRenamed('Handcap', 'Handicap')
df.display()

In [0]:
df = df.withColumn('Scheduled_Day', col('Scheduled_Day').cast('timestamp'))\
    .withColumn('Appointment_Day', col('Appointment_Day').cast('timestamp'))
df.display()

In [0]:
df = df.withColumn('invalid_age', \
    when((col("Age") < 0)| (col("Age") > 100), True).otherwise(False))\
    .withColumn('invalid_date_diff', when(col("Scheduled_Day") > col("Appointment_Day"), True).otherwise(False))
df.display()

In [0]:
df.groupBy('Appointment_ID').count().filter(col('count') > 1).display()
df.filter(col('Appointment_ID').isNull()).display()

In [0]:
df = df.withColumn('NoShow', col('NoShow').cast('boolean'))
df.display()

In [0]:
df = df.withColumnRenamed('NoShow','Missed_Appointment')
df.display()


In [0]:
df = df.withColumn('wait_days', (to_date("Appointment_Day") - to_date("Scheduled_Day")).cast('int'))
df.display()

In [0]:
df = df.withColumn("age_group", \
    when((col("age")).isNull(), "Unknown")\
    .when((col("age") ) < 0, "Invalid_age")\
    .when((col("age")) <18, "Under 18")\
    .when((col("age")) <36, "18-35")\
    .when((col("age")) <51, "36-50")\
    .when((col("age")) <66, "51-65")\
    .otherwise("Above 65"))
df.display() 

In [0]:
df = df.withColumn('appointment_weekday', (date_format(col('Appointment_Day'), 'EEEE')))
df.display()

In [0]:
output_path = "/Volumes/workspace/default/my_files/medical_appointment/transform"

df.write.mode("overwrite").parquet(output_path)

### AGGREGATE

#### SUMMARY

No-show rate by age group
No-show rate by neighbourhood
No-show rate by wait time bucket (same day, 1 to 7 days, 8 to 30 days, 30+)
No-show rate by SMS reminder received vs not
No-show rate by scholarship (welfare programme) status

In [0]:
no_show_rate_by_age_group = (df.groupBy('age_group')\
    .agg(\
        (avg(col('Missed_Appointment').cast('int'))*100)\
            .alias('no_show_rate')))
no_show_rate_by_age_group.display()

In [0]:
no_show_rate_by_age_group = (df.groupBy('neighbourhood')\
    .agg(\
        (avg(col('Missed_Appointment').cast('int'))*100)\
            .alias('no_show_rate')))
no_show_rate_by_age_group.display()

In [0]:
def calculate_no_show_rate(df, group_column):
    return (
        df.groupBy(group_column)
        .agg(
            count('*').alias('total_appointments'),
            avg(
                col('Missed_Appointment').cast('int')
            ).alias('no_show_rate')
        )
        .withColumn(
            'no_show_rate',
            col('no_show_rate') * 100
        )
    )

In [0]:
gold_no_show_by_age = calculate_no_show_rate(
    df, 'age_group'
)
gold_no_show_by_age.display()

In [0]:
gold_no_show_by_neighbourhood = calculate_no_show_rate(
    df, 'Neighbourhood'
)
gold_no_show_by_neighbourhood.display()

In [0]:
gold_no_show_by_wait_time = calculate_no_show_rate(
    df, 'wait_days'
).orderBy(col('wait_days').asc())
gold_no_show_by_wait_time.display()

In [0]:
gold_no_show_by_sms = calculate_no_show_rate(
    df, 'SMS_received'
)
gold_no_show_by_sms.display()

In [0]:
gold_no_show_by_scholarship = calculate_no_show_rate(
    df, 'Scholarship'
)
gold_no_show_by_scholarship.display()

In [0]:
gold_no_show_by_age.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable('gold_no_show_by_age')

### LOAD

In [0]:
gold_parquet_config = {
    "gold_no_show_by_age": (
        gold_no_show_by_age,
        "age_group"
    ),
    "gold_no_show_by_wait_time": (
        gold_no_show_by_wait_time,
        "wait_days"
    ),
    "gold_no_show_by_sms": (
        gold_no_show_by_sms,
        "SMS_received"
    ),
    "gold_no_show_by_scholarship": (
        gold_no_show_by_scholarship,
        "Scholarship"
    )
}

base_path = (
    "/Volumes/workspace/default/my_files/"
    "medical_appointment/gold"
)

for table_name, (table_df, partition_column) in gold_parquet_config.items():
    (
        table_df.write
        .format("parquet")
        .mode("overwrite")
        .partitionBy(partition_column)
        .save(f"{base_path}/{table_name}")
    )

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/my_files/medical_appointment/"
    )
)